In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install soundfile

import soundfile as sf
import numpy as np


Mounted at /content/drive


In [ ]:
!pip -q install librosa


In [ ]:

import math
import re
import random
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def load_wav_mono(path: Path, target_sr: int = 16000) -> torch.Tensor:
    """
    Loads wav using soundfile, converts to mono float32, resamples with scipy if needed.
    Returns torch tensor [N] float32.
    """
    y, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)

    if sr != target_sr:
        g = math.gcd(sr, target_sr)
        up = target_sr // g
        down = sr // g
        y = resample_poly(y, up, down).astype(np.float32)

    mx = np.max(np.abs(y)) + 1e-8
    y = (y / mx).astype(np.float32)
    return torch.from_numpy(y)


Device: cpu


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/PiENet_data")
SPEECH_ROOT = DATA_ROOT / "SPEECH DATA"

INPUT_CATEGORY = "MIC"
GENDERS = ["FEMALE", "MALE"]

def collect_pairs_structured(speech_root: Path, genders, input_cat: str):
    pairs = []
    missing = 0
    for gender in genders:
        in_dir  = speech_root / gender / input_cat
        ref_dir = speech_root / gender / "REF"

        wavs = list(in_dir.rglob("*.wav"))
        for w in wavs:
            if input_cat.upper() == "MIC":
                ref_name = w.name.replace("mic_", "ref_").replace(".wav", ".f0")
            elif input_cat.upper() == "LAR":
                ref_name = w.name.replace("lar_", "ref_").replace(".wav", ".f0")
            else:
                raise ValueError("INPUT_CATEGORY must be 'MIC' or 'LAR'")

            r = ref_dir / w.parent.name / ref_name
            if r.exists():
                pairs.append((w, r))
            else:
                missing += 1

    print(f"Paired utterances: {len(pairs)} | Missing REF: {missing}")
    return pairs

pairs = collect_pairs_structured(SPEECH_ROOT, GENDERS, INPUT_CATEGORY)

for i in range(min(3, len(pairs))):
    print(pairs[i][0])
    print(pairs[i][1])
    print("---")


Paired utterances: 4718 | Missing REF: 0
/content/drive/MyDrive/PiENet_data/SPEECH DATA/FEMALE/MIC/F07/mic_F07_sx276.wav
/content/drive/MyDrive/PiENet_data/SPEECH DATA/FEMALE/REF/F07/ref_F07_sx276.f0
---
/content/drive/MyDrive/PiENet_data/SPEECH DATA/FEMALE/MIC/F07/mic_F07_sx277.wav
/content/drive/MyDrive/PiENet_data/SPEECH DATA/FEMALE/REF/F07/ref_F07_sx277.f0
---
/content/drive/MyDrive/PiENet_data/SPEECH DATA/FEMALE/MIC/F07/mic_F07_sx287.wav
/content/drive/MyDrive/PiENet_data/SPEECH DATA/FEMALE/REF/F07/ref_F07_sx287.f0
---


In [ ]:
def speaker_id_from_filename(wav_path: Path):
    m = re.search(r"_(F\d\d|M\d\d)_", wav_path.name, flags=re.IGNORECASE)
    return m.group(1).upper() if m else wav_path.parent.name.upper()

spk_to_items = {}
for w, r in pairs:
    spk = speaker_id_from_filename(w)
    spk_to_items.setdefault(spk, []).append((w, r))

speakers = sorted(spk_to_items.keys())
random.seed(0)
random.shuffle(speakers)

n = len(speakers)
train_spk = set(speakers[: int(0.8*n)])
val_spk   = set(speakers[int(0.8*n): int(0.9*n)])
test_spk  = set(speakers[int(0.9*n):])

train_pairs = [x for s in train_spk for x in spk_to_items[s]]
val_pairs   = [x for s in val_spk   for x in spk_to_items[s]]
test_pairs  = [x for s in test_spk  for x in spk_to_items[s]]

print("Speakers:", n)
print("Train/Val/Test utterances:", len(train_pairs), len(val_pairs), len(test_pairs))
print("Train spk:", sorted(train_spk))
print("Val spk:", sorted(val_spk))
print("Test spk:", sorted(test_spk))


Speakers: 20
Train/Val/Test utterances: 3774 472 472
Train spk: ['F01', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F10', 'M01', 'M02', 'M05', 'M06', 'M07', 'M08', 'M09', 'M10']
Val spk: ['F02', 'F09']
Test spk: ['M03', 'M04']


In [ ]:
# Framing (10 ms hop at 16 kHz -> 160 samples)
TARGET_SR = 16000
WINLEN = 512
HOP = 160
HOP_SEC = HOP / TARGET_SR

# PiENet-like classification bins
NBINS = 351
F0_MIN = 50.0
F0_MAX = 500.0

def make_f0_bins(nbins=NBINS, f0_min=F0_MIN, f0_max=F0_MAX):
    voiced_bins = np.exp(np.linspace(np.log(f0_min), np.log(f0_max), nbins-1)).astype(np.float32)
    bins = np.concatenate([[0.0], voiced_bins]).astype(np.float32)  # bin 0 = unvoiced
    return bins

F0_BINS = make_f0_bins()

def load_ref_f0_ascii(f0_path: Path) -> np.ndarray:
    arr = np.loadtxt(str(f0_path))
    return arr[:, 0].astype(np.float32)  # col1 = F0 Hz, 0 = unvoiced

def f0_to_class_idx(f0_hz: np.ndarray, bins: np.ndarray) -> np.ndarray:
    idx = np.zeros_like(f0_hz, dtype=np.int64)
    voiced = f0_hz > 0
    if np.any(voiced):
        diffs = np.abs(np.log(f0_hz[voiced, None] + 1e-8) - np.log(bins[None, 1:] + 1e-8))
        idx_voiced = 1 + np.argmin(diffs, axis=1)
        idx[voiced] = idx_voiced
    return idx

def frame_signal_pienet(y_1d: torch.Tensor, winlen=WINLEN, hop=HOP) -> torch.Tensor:
    """
    PiENet-like convention:
      - pad winlen/2 on both ends
      - n_frames = ceil(original_length / hop)
    """
    assert y_1d.ndim == 1
    N = y_1d.numel()
    pad = winlen // 2
    y = F.pad(y_1d, (pad, pad))
    frames_all = y.unfold(0, winlen, hop)  # [~, winlen]
    n_frames = int(math.ceil(N / hop))
    return frames_all[:n_frames]

def add_gaussian_noise(y: torch.Tensor, snr_db_low=0.0, snr_db_high=20.0) -> torch.Tensor:
    snr_db = float(torch.empty(1).uniform_(snr_db_low, snr_db_high).item())
    sig_power = y.pow(2).mean().clamp_min(1e-8)
    noise = torch.randn_like(y)
    noise_power = noise.pow(2).mean().clamp_min(1e-8)
    scale = torch.sqrt(sig_power / (noise_power * (10 ** (snr_db / 10))))
    return y + scale * noise

class PTDBPiENetDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        wav_path, f0_path = self.pairs[idx]

        wav = load_wav_mono(wav_path, TARGET_SR)  # torch [N]

        if self.augment:
            wav = add_gaussian_noise(wav, 0.0, 20.0)
            gain = float(torch.empty(1).uniform_(0.7, 1.2).item())
            wav = (gain * wav).clamp(-1.0, 1.0)

        frames = frame_signal_pienet(wav)         # [T, 512]

        f0_ref = load_ref_f0_ascii(f0_path)       # np [Tref]
        cls = f0_to_class_idx(f0_ref, F0_BINS)    # np int64 [Tref]

        T = min(frames.shape[0], cls.shape[0])
        frames = frames[:T].float()
        cls_t = torch.from_numpy(cls[:T]).long()
        f0_t  = torch.from_numpy(f0_ref[:T]).float()

        return frames, cls_t, f0_t, str(wav_path)

def collate_batch(batch):
    frames, cls, f0_ref, names = zip(*batch)
    lengths = torch.tensor([x.shape[0] for x in frames], dtype=torch.long)
    T_max = int(lengths.max().item())

    frames_p = torch.zeros((len(batch), T_max, WINLEN), dtype=torch.float32)
    cls_p = torch.full((len(batch), T_max), -100, dtype=torch.long)
    f0_p  = torch.zeros((len(batch), T_max), dtype=torch.float32)

    for i, (fr, cl, f0) in enumerate(zip(frames, cls, f0_ref)):
        T = fr.shape[0]
        frames_p[i, :T] = fr
        cls_p[i, :T] = cl
        f0_p[i, :T] = f0

    return frames_p, cls_p, f0_p, lengths, names

BATCH_SIZE = 8
train_loader = DataLoader(PTDBPiENetDataset(train_pairs, augment=True),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader   = DataLoader(PTDBPiENetDataset(val_pairs, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader  = DataLoader(PTDBPiENetDataset(test_pairs, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)

print("Batch check:", next(iter(train_loader))[0].shape)  # [B,T,512]


Batch check: torch.Size([8, 997, 512])


In [ ]:
class PiENetLike(nn.Module):
    def __init__(self,
                 winlen=WINLEN,
                 nbins=NBINS,
                 residual_channels=128,
                 filter_width=5,
                 dilations=(1,2,4,8,1,2,4,8),
                 postnet_channels=256,
                 dropout=0.4):
        super().__init__()
        self.in_proj = nn.Conv1d(winlen, residual_channels, kernel_size=1)
        self.dropout = nn.Dropout(p=dropout)

        self.fg_convs = nn.ModuleList([
            nn.Conv1d(residual_channels, 2*residual_channels,
                      kernel_size=filter_width, dilation=d,
                      padding=(filter_width//2)*d)
            for d in dilations
        ])
        self.skip_convs = nn.ModuleList([nn.Conv1d(residual_channels, residual_channels, kernel_size=1) for _ in dilations])
        self.out_convs  = nn.ModuleList([nn.Conv1d(residual_channels, residual_channels, kernel_size=1) for _ in dilations])

        self.post1 = nn.Conv1d(residual_channels, postnet_channels,
                               kernel_size=filter_width, padding=filter_width//2)
        self.post2 = nn.Conv1d(postnet_channels, nbins,
                               kernel_size=filter_width, padding=filter_width//2)

    def forward(self, frames_bt):
        # frames_bt: [B,T,512]
        x = frames_bt.transpose(1, 2)     # [B,512,T]
        x = self.dropout(x)
        r = torch.tanh(self.in_proj(x))   # [B,R,T]

        skips = []
        h = r
        for fg, sk, out in zip(self.fg_convs, self.skip_convs, self.out_convs):
            y = fg(h)                     # [B,2R,T]
            a, b = y.chunk(2, dim=1)
            y = torch.tanh(a) * torch.sigmoid(b)
            skips.append(sk(y))           # [B,R,T]
            h = out(y) + h                # residual

        s = torch.stack(skips, dim=0).sum(dim=0)  # [B,R,T]
        z = torch.relu(self.post1(s))
        logits = self.post2(z)            # [B,NBINS,T]
        return logits.transpose(1, 2)     # [B,T,NBINS]

model = PiENetLike().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

print(model)


PiENetLike(
  (in_proj): Conv1d(512, 128, kernel_size=(1,), stride=(1,))
  (dropout): Dropout(p=0.4, inplace=False)
  (fg_convs): ModuleList(
    (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(4,), dilation=(2,))
    (2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(8,), dilation=(4,))
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(16,), dilation=(8,))
    (4): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(4,), dilation=(2,))
    (6): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(8,), dilation=(4,))
    (7): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(16,), dilation=(8,))
  )
  (skip_convs): ModuleList(
    (0-7): 8 x Conv1d(128, 128, kernel_size=(1,), stride=(1,))
  )
  (out_convs): ModuleList(
    (0-7): 8 x Conv1d(128, 128, kernel_size=(1,), stride=(1,))
  )
  (po

In [ ]:
from tqdm import tqdm

SAVE_DIR = Path("/content/drive/MyDrive/PiENet_colab/models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = SAVE_DIR / f"pienetlike_mic_{'_'.join([g.lower() for g in GENDERS])}_bins{NBINS}_{int(F0_MIN)}to{int(F0_MAX)}.pt"

def make_time_mask(lengths, T):
    idx = torch.arange(T, device=lengths.device)[None, :]
    return idx < lengths[:, None]

best_val = float("inf")
EPOCHS = 40

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for frames, cls, f0_ref, lengths, _ in tqdm(train_loader, desc=f"Epoch {epoch} train"):
        frames = frames.to(device)
        cls = cls.to(device)

        logits = model(frames)  # [B,T,C]
        B, T, C = logits.shape
        loss = criterion(logits.reshape(B*T, C), cls.reshape(B*T))

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        opt.step()
        tr_loss += loss.item()

    tr_loss /= max(1, len(train_loader))
    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for frames, cls, f0_ref, lengths, _ in tqdm(val_loader, desc=f"Epoch {epoch} val"):
            frames = frames.to(device)
            cls = cls.to(device)

            logits = model(frames)
            B, T, C = logits.shape
            loss = criterion(logits.reshape(B*T, C), cls.reshape(B*T))
            va_loss += loss.item()

    va_loss /= max(1, len(val_loader))
    print(f"Epoch {epoch}: train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}")

    if va_loss < best_val:
        best_val = va_loss
        torch.save({
            "model_state": model.state_dict(),
            "bins": F0_BINS,
            "target_sr": TARGET_SR,
            "winlen": WINLEN,
            "hop": HOP,
            "nbins": NBINS,
            "f0_min": F0_MIN,
            "f0_max": F0_MAX,
            "genders": GENDERS,
            "input_category": "MIC",
        }, SAVE_PATH)
        print("Saved best:", SAVE_PATH)
